# Assess the fine-tuned Cellpose model

This notebook compares the default Cellpose-SAM model with the fine-tuned model using aggregate segmentation metrics. The output is limited to summary tables; no per-image results are displayed.

> **Evaluation note:** the `test` split was passed to `train.train_seg` in the training notebook for loss monitoring. Results here are useful for comparison, but are not an independent final test estimate.

## Configure paths and load the test split

The model name below matches the current training settings in `02_train_model.ipynb`. Update it if the training parameters or model name change.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from cellpose import core, io, metrics, models

dataset_name = "A172_splits_n100_seed42"
model_name = (
    "A172_splits_n100_seed42_cpsam_v2_finetuned_"
    "n_epochs100_b_size1_w_decay0.1_l_rate1e-05"
)
iou_thresholds = [0.5, 0.75, 0.9]

data_dir_candidates = (
    Path("../data"),
    Path("data"),
    Path("cellpose/data"),
    Path("cellpose/scripts/../data"),
)
data_dir = next(
    (path for path in data_dir_candidates if (path / dataset_name / "test").is_dir()),
    None,
)
if data_dir is None:
    raise FileNotFoundError(
        f"Could not find test split for {dataset_name}; checked: {data_dir_candidates}"
    )
test_dir = data_dir / dataset_name / "test"

test_image_paths = sorted(test_dir.glob("*.tif"))
if not test_image_paths:
    raise FileNotFoundError(f"No .tif test images found in {test_dir.resolve()}")

test_mask_paths = [test_dir / f"{path.stem}_seg.npy" for path in test_image_paths]
missing_masks = [path for path in test_mask_paths if not path.is_file()]
if missing_masks:
    raise FileNotFoundError(f"Missing ground-truth mask: {missing_masks[0].resolve()}")

test_images = [io.imread(str(path)) for path in test_image_paths]
test_labels = [np.load(path, allow_pickle=True).item()["masks"] for path in test_mask_paths]
for image_path, image, label in zip(test_image_paths, test_images, test_labels):
    if image.shape[:2] != label.shape[:2]:
        raise ValueError(
            f"Image/mask shape mismatch for {image_path}: {image.shape} vs {label.shape}"
        )

print(f"Loaded {len(test_images)} test images and ground-truth masks from {test_dir.resolve()}")

Loaded 10 test images and ground-truth masks from /storage/research/dsl_vibe_rs/trainings/2026_deep_learning_with_VIBE/cellpose/data/A172_splits_n100_seed42/test


## Load the models and generate predictions

In [2]:
io.logger_setup()
if not core.use_gpu():
    raise ImportError("No GPU access; use a VIBE instance with GPU support")

model_path_candidates = (
    Path("models") / model_name,
    Path("models") / f"{model_name}.npy",
    Path("../scripts/models") / model_name,
    Path("../scripts/models") / f"{model_name}.npy",
    Path("cellpose/scripts/models") / model_name,
    Path("cellpose/scripts/models") / f"{model_name}.npy",
)
trained_model_path = next((path for path in model_path_candidates if path.exists()), None)
if trained_model_path is None:
    raise FileNotFoundError(
        "Could not find the fine-tuned model. Update model_name or check these paths: "
        f"{model_path_candidates}"
    )

default_model = models.CellposeModel(gpu=True)
fine_tuned_model = models.CellposeModel(
    gpu=True,
    pretrained_model=str(trained_model_path),
)
print(f"Loaded fine-tuned model: {trained_model_path.resolve()}")

default_predictions = [default_model.eval(image)[0] for image in test_images]
fine_tuned_predictions = [fine_tuned_model.eval(image)[0] for image in test_images]

[GUI INFO] : WRITING LOG OUTPUT TO /storage/homefs/rl18y689/.cellpose/run.log

cellpose version: 	4.2.1.1 
platform:       	linux 
python version: 	3.13.15 
torch version:  	2.14.0+cu130
2026-10-26 10:35:23,862 [io INFO] WRITING LOG OUTPUT TO /storage/homefs/rl18y689/.cellpose/run.log
2026-10-26 10:35:23,863 [io INFO] 
cellpose version: 	4.2.1.1 
platform:       	linux 
python version: 	3.13.15 
torch version:  	2.14.0+cu130
2026-10-26 10:35:24,124 [core INFO] ** TORCH CUDA version installed and working. **
2026-10-26 10:35:24,125 [core INFO] ** TORCH CUDA version installed and working. **
2026-10-26 10:35:24,125 [core INFO] >>>> using GPU (CUDA)
2026-10-26 10:35:31,039 [models INFO] >>>> loading model /storage/homefs/rl18y689/.vibe/.cellpose/models/cpsam_v2
2026-10-26 10:35:31,612 [core INFO] ** TORCH CUDA version installed and working. **
2026-10-26 10:35:31,612 [core INFO] >>>> using GPU (CUDA)
2026-10-26 10:35:38,228 [models INFO] >>>> loading model models/A172_splits_n100_seed42_c

/opt/.pixi/envs/cellpose-gui-training/lib/python3.13/site-packages/cellpose/dynamics.py:541: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:848.)
  coo = torch.sparse_coo_tensor(pt, torch.ones(pt.shape[1], device=pt.device, dtype=torch.int),


## Summarize segmentation performance

### What the metrics mean

- **Intersection over Union (IoU):** overlap between a predicted object mask and its ground-truth mask, calculated as intersection area divided by union area. A predicted/true object pair counts as a match only if its IoU reaches the selected threshold.
- **Mean AP @ 0.50 / 0.75 / 0.90:** in Cellpose, AP is the object-level score `TP / (TP + FP + FN)` at the specified IoU threshold. The notebook computes it for each image and averages those scores across images. Higher is better. This is Cellpose's thresholded mask-matching AP, not the area under a confidence-ranked precision-recall curve.
- **Precision @ 0.50:** `TP / (TP + FP)`, using TP and FP summed over all test images. Of the predicted objects, this is the fraction matched to ground truth. Higher means fewer unmatched/extra predictions.
- **Recall @ 0.50:** `TP / (TP + FN)`, using counts summed over the test images. Of the ground-truth objects, this is the fraction detected. Higher means fewer missed objects.
- **F1 @ 0.50:** harmonic mean of the reported precision and recall, `2 * precision * recall / (precision + recall)`. It balances missed and extra objects; higher is better.
- **Mean absolute object-count error:** average across images of `abs(number of predicted objects - number of ground-truth objects)`. Lower is better, but it does not measure whether object boundaries or identities match.

These metrics summarize different properties and should be interpreted together. All matching metrics use IoU thresholds, so a visually close mask may still count as a miss if its overlap falls below the threshold.

### References

1. Cellpose `average_precision` implementation and IoU-based object matching: [MouseLand/cellpose, `cellpose/metrics.py`](https://github.com/MouseLand/cellpose/blob/main/cellpose/metrics.py).
2. Stringer C, Wang T, Michaelos M, Pachitariu M. *Cellpose: a generalist algorithm for cellular segmentation.* Nature Methods 18, 100-106 (2021). [https://doi.org/10.1038/s41592-020-01018-x](https://doi.org/10.1038/s41592-020-01018-x).

The table contains only model-level summary values; no per-image metric table is generated.

In [3]:
def summarize_predictions(model_label, predictions):
    if len(predictions) != len(test_labels):
        raise ValueError(
            f"{model_label} produced {len(predictions)} masks for {len(test_labels)} test labels"
        )

    ap, true_positive, false_positive, false_negative = metrics.average_precision(
        test_labels,
        predictions,
        threshold=iou_thresholds,
    )
    tp = true_positive[:, 0].sum()
    fp = false_positive[:, 0].sum()
    fn = false_negative[:, 0].sum()
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    true_object_counts = [len(np.unique(mask)) - 1 for mask in test_labels]
    predicted_object_counts = [len(np.unique(mask)) - 1 for mask in predictions]

    return {
        "Model": model_label,
        "Images": len(test_labels),
        "Mean AP @ 0.50": ap[:, 0].mean(),
        "Mean AP @ 0.75": ap[:, 1].mean(),
        "Mean AP @ 0.90": ap[:, 2].mean(),
        "Precision @ 0.50": precision,
        "Recall @ 0.50": recall,
        "F1 @ 0.50": f1,
        "Mean abs. object-count error": np.mean(
            np.abs(np.asarray(predicted_object_counts) - np.asarray(true_object_counts))
        ),
    }


summary = pd.DataFrame(
    [
        summarize_predictions("Default Cellpose-SAM", default_predictions),
        summarize_predictions("Fine-tuned Cellpose-SAM", fine_tuned_predictions),
    ]
)
metric_columns = summary.columns.drop(["Model", "Images"])
summary[metric_columns] = summary[metric_columns].astype(float).round(3)
display(summary)

,Model,Images,Mean AP @ 0.50,Mean AP @ 0.75,Mean AP @ 0.90,Precision @ 0.50,Recall @ 0.50,F1 @ 0.50,Mean abs. object-count error
0,Default Cellpose-SAM,10,0.540,0.306,0.051,0.586,0.598,0.592,35.9
1,Fine-tuned Cellpose-SAM,10,0.529,0.302,0.052,0.646,0.533,0.584,41.9
